## Overview of GitHub Repository

 ### Directory Structure and Files

Please note that I only record the files that I have seen and think are important. There are directories that are not there at the beginning, but created during the execution of the program, or you have to create them.

```
PufferDrive/
├── config/
├── data/
│   │   └── processed/
│   │   │   └── training/
├── experiments/
├── pufferlib/
│   ├── config/
│   │   ├── ocean/
│   │   │   └── drive.ini
│   │   └── default.ini
│   ├── ocean/
│   │   ├── drive/
│   │   │   ├── binding.c
│   │   │   ├── drive.c
│   │   │   └── drive.py
│   │   ├── __init__.py
│   │   ├── env_binding.h
│   │   ├── environment.py
│   │   └── torch.py
│   ├── resources/
│   │   └── drive/
│   │   │   └── binaries/
│   └── pufferl.py
├── resources/
└── setup.py
```


### Explanation

**PufferDrive/config/**
- it should be a mirror of **PufferDrive/pufferlib/config/**

**PufferDrive/data/processed/training/**
- you need to create this folder by yourself
- it should store the raw json file map data you downloaded

**PufferDrive/experiments/**
- this directory should be automatically created if not exist when you run "*puffer train puffer_drive ...*"
- it stores the current checkpoint and final model

**PufferDrive/pufferlib/config/ocean/drive.ini**
- configurations specified for the Drive project
- it will overwrite the configurations in **PufferDrive/pufferlib/config/default.ini** if there are same config variable under the same section

**PufferDrive/pufferlib/config/default.ini**
- default configurations

**PufferDrive/pufferlib/ocean/drive/binding.c**
- the main C file imported by **PufferDrive/pufferlib/ocean/drive/drive.py**
- it imports other C files like **PufferDrive/pufferlib/ocean/drive/drive.c** (that imports other C files in the same directory) and **PufferDrive/pufferlib/ocean/env_binding.h**
- it serves as the driving simulator

**PufferDrive/pufferlib/ocean/drive/drive.c**
- A C file imported by **PufferDrive/pufferlib/ocean/drive/binding.c**

**PufferDrive/pufferlib/ocean/drive/drive.py**
- the script mainly does two things: defines a drive environments class and processes raw json files to binary files.
- Class Drive:
  - wraps binding functions from **PufferDrive/pufferlib/ocean/drive/drive.c** to build drive environments
  - create drive environments using binary data from **PufferDrive/pufferlib/resources/drive/binaries**
- Function process_all_maps():
  - process all raw json files from **PufferDrive/data/processed/training/** and save them to **PufferDrive/pufferlib/resources/drive/binaries**
  - this function will be executed when you call "*python pufferlib/ocean/drive/drive.py*"

**PufferDrive/pufferlib/ocean/__init__.py**
  - imported by **PufferDrive/pufferlib/pufferl.py** through Function load_env() and load_policy()
  - it imports **PufferDrive/pufferlib/ocean/environment.py** and **PufferDrive/pufferlib/ocean/torch.py**

**PufferDrive/pufferlib/ocean/env_binding.h**:
  - contains many binding functions that are used in **PufferDrive/pufferlib/ocean/drive/drive.py**

**PufferDrive/pufferlib/ocean/environment.py**
  - Function env_creator():
    - get the Class Drive from **PufferDrive/pufferlib/ocean/drive/drive.py**

**PufferDrive/pufferlib/ocean/torch.py**
  - it defines a PPO framework class. It is much more complicated than the example we had, but it also output action and value as regular PPO.
  - Class Drive:
    - Unlike the Class Drive in **PufferDrive/pufferlib/ocean/drive/drive.py**, this is the PPO framework

**PufferDrive/pufferlib/resources/drive/binaries**
  - this is where the binary data is stored

**PufferDrive/pufferlib/pufferl.py**
  - this is the main program that will be executed when you run "*puffer [train, eval] puffer_drive ...*" (console script?)
  - Function train():
    - it will be executed when you run "*puffer train puffer_drive ...*"
    - it will train a PPO model (Class PuffeRL) from scratch (it should be able to continue training an existing model if you provide "*load-model-path*")
  - Function eval():
    - it will be executed when you run "*puffer eval puffer_drive ...*"
    - note that it will use the same data (**PufferDrive/pufferlib/resources/drive/binaries**) as Function train() due to how Class Drive (the one from **PufferDrive/pufferlib/ocean/drive/drive.py**) is defined
  - class PuffeRL:
    - this is the main class that wraps the entire PPO model
    - note that there is a Function train() and evaluate() within the Class, they are different from the Function mentioned above
  - Function load_env():
    - imports **PufferDrive/pufferlib/ocean/__init__.py**, which imports **PufferDrive/pufferlib/ocean/environment.py**, to use Function env_creator() to get Class Drive from **PufferDrive/pufferlib/ocean/drive/drive.py**
    - creates vectorized environments
  - Function load_policy():
    - imports **PufferDrive/pufferlib/ocean/__init__.py**, which imports the Class Drive from **PufferDrive/pufferlib/ocean/torch.py**
    - create an instance of the PPO model
    - it will load the state dictionary of an existing model if you specified "*load-model-path*"

**PufferDrive/resources/**
- it should be a mirror of **PufferDrive/pufferlib/resources/**

**PufferDrive/setup.py**
- set up PufferDrive
- it should enable **PufferDrive/pufferlib/pufferl.py** as console script?

## Example

### Setup

- I'm not sure what r62.tar.gz do
- Please be sure you are under "PufferDrive", it is required for all steps

In [1]:
import os, getpass, pathlib
import subprocess, sys
from pathlib import Path

user = getpass.getuser()
scratch_root = os.environ.get("SCRATCH", f"/scratch/{user}")
SRC = pathlib.Path(scratch_root) / "src"
DATA = pathlib.Path(scratch_root) / "data"
TMP  = pathlib.Path(scratch_root) / "tmp"
CACHE = pathlib.Path(scratch_root) / ".cache"

for p in (SRC, DATA, TMP, CACHE):
    p.mkdir(parents=True, exist_ok=True)

# Optional: keep caches off $HOME for builds/pip
os.environ["TMPDIR"] = str(TMP)
os.environ["XDG_CACHE_HOME"] = str(CACHE)
os.environ["PIP_CACHE_DIR"] = str(CACHE / "pip")

SRC

PosixPath('/scratch/fz2411/src')

In [2]:
'''
subprocess.run(
    ["git", "clone", "--recursive", "https://github.com/Emerge-Lab/PufferDrive.git"],
    cwd=str(SRC), check=True
)
'''

'\nsubprocess.run(\n    ["git", "clone", "--recursive", "https://github.com/Emerge-Lab/PufferDrive.git"],\n    cwd=str(SRC), check=True\n)\n'

In [3]:
# Download into SRC (or change to DATA if you prefer)
!wget -c -P "{SRC}" https://github.com/benhoyt/inih/archive/r62.tar.gz
!tar -xzf "{SRC}/r62.tar.gz" -C "{SRC}"
# Optional: move into a third_party folder inside PufferDrive if needed
!(mkdir -p "{SRC}/PufferDrive/third_party" && mv "{SRC}/inih-r62" "{SRC}/PufferDrive/third_party/inih")

--2025-10-30 13:42:20--  https://github.com/benhoyt/inih/archive/r62.tar.gz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/benhoyt/inih/tar.gz/refs/tags/r62 [following]
--2025-10-30 13:42:21--  https://codeload.github.com/benhoyt/inih/tar.gz/refs/tags/r62
Resolving codeload.github.com (codeload.github.com)... 140.82.113.10
Connecting to codeload.github.com (codeload.github.com)|140.82.113.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘/scratch/fz2411/src/r62.tar.gz’

r62.tar.gz              [ <=>                ]  21.63K  --.-KB/s    in 0.006s  

2025-10-30 13:42:21 (3.72 MB/s) - ‘/scratch/fz2411/src/r62.tar.gz’ saved [22145]

mv: cannot move '/scratch/fz2411/src/inih-r62' to '/scratch/fz2411/src/PufferDrive/third_party/inih/inih-r62': File exists


In [4]:
repo_path = SRC / "PufferDrive"
%pip install -U pip setuptools wheel
%pip install -e {repo_path}
# If build isolation causes issues on HPC:
# %pip install -e {repo_path} --no-build-isolation -v


Note: you may need to restart the kernel to use updated packages.
Obtaining file:///scratch/fz2411/src/PufferDrive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pufferlib (pyproject.toml) ... done
  Created wheel for pufferlib: filename=pufferlib-3.0.0-0.editable-cp311-cp311-linux_x86_64.whl size=7823 sha256=58b3661942b7942b06d54383c0ff4b7cef5ca72f1d40eee79c03acc1a5bd0d6f
  Stored in directory: /scratch/fz2411/tmp/pip-ephem-wheel-cache-7or_yixl/wheels/2c/47/07/2788c5b03cb638a8c847ec21f86f8dd5312a11f2077b34648c
Successfully built pufferlib
  Attempting uninstall: pufferlib
    Found existing installation: pufferlib 3.0.0
    Uninstalling pufferlib-3.0.0:
      Successfully uninstalled pufferlib-3.0.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -U pip setuptools wheel
%pip install -U ninja cmake

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Set project path (adjust if needed)
project_path = Path("/scratch/fz2411/src/PufferDrive")

# Change working directory
os.chdir(project_path)

# Check if setup.py exists
assert (project_path / "setup.py").exists(), "setup.py not found in project path."

# Run the build_ext command using the current kernel's Python
!{sys.executable} setup.py build_ext --inplace --force

/scratch/fz2411/puffer_venv/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
!!

        ********************************************************************************
        Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).

        By 2026-Feb-18, you need to update your project and remove deprecated calls
        or your builds will no longer be supported.

        See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
        ********************************************************************************

!!
  corresp(dist, value, root_dir)
running build_ext
running build_torch
W1030 13:44:25.012000 2764280 torch/utils/cpp_extension.py:630] Attempted to use ninja as the BuildExtension backend but we could not find ninja

In [7]:
!{sys.executable} -m pufferlib.pufferl train puffer_drive --help

Usage: pufferl.py [--load-model-path LOAD_MODEL_PATH] [--load-id LOAD_ID]
                  [--render-mode {auto,human,ansi,rgb_array,raylib,None}]
                  [--save-frames SAVE_FRAMES] [--gif-path GIF_PATH]
                  [--fps FPS] [--max-runs MAX_RUNS] [--wandb]
                  [--wandb-project WANDB_PROJECT] [--wandb-group WANDB_GROUP]
                  [--neptune] [--neptune-name NEPTUNE_NAME]
                  [--neptune-project NEPTUNE_PROJECT]
                  [--local-rank LOCAL_RANK] [--tag TAG] [--package PACKAGE]
                  [--env-name ENV_NAME] [--policy-name POLICY_NAME]
                  [--rnn-name RNN_NAME]
                  [--max-suggestion-cost MAX_SUGGESTION_COST]
                  [--vec.backend VEC.BACKEND] [--vec.num-envs VEC.NUM_ENVS]
                  [--vec.num-workers VEC.NUM_WORKERS]
                  [--vec.batch-size VEC.BATCH_SIZE]
                  [--vec.zero-copy VEC.ZERO_COPY] [--vec.seed VEC.SEED]
                  [--env.num-a

### Data Preparation

- you need to create the directory **PufferDrive/data/processed/training/** and copy all training data to this folder because **PufferDrive/pufferlib/ocean/drive/drive.py** will look for this directory as explained above

In [8]:
!git clone https://huggingface.co/datasets/EMERGE-lab/GPUDrive_mini

fatal: destination path 'GPUDrive_mini' already exists and is not an empty directory.


In [9]:
!mkdir -p data/processed/training

In [10]:
!cp -a GPUDrive_mini/training/. data/processed/training

In [11]:
import sys
# This guarantees you use the same interpreter the notebook is using:
!{sys.executable} -m pufferlib.ocean.drive.drive --help

Found 1150 JSON files
Processing tfrecord-00000-of-00150_135.json -> map_000.bin
23
172
Processing tfrecord-00000-of-00150_254.json -> map_001.bin
9
741
Processing tfrecord-00000-of-01000_401.json -> map_002.bin
64
38
Processing tfrecord-00002-of-00150_31.json -> map_003.bin
30
110
Processing tfrecord-00002-of-01000_321.json -> map_004.bin
20
268
Processing tfrecord-00002-of-01000_345.json -> map_005.bin
364
106
Processing tfrecord-00004-of-01000_282.json -> map_006.bin
74
303
Processing tfrecord-00004-of-01000_306.json -> map_007.bin
62
39
Processing tfrecord-00004-of-01000_428.json -> map_008.bin
110
244
Processing tfrecord-00004-of-01000_64.json -> map_009.bin
178
323
Processing tfrecord-00005-of-00150_84.json -> map_010.bin
18
65
Processing tfrecord-00005-of-01000_117.json -> map_011.bin
53
114
Processing tfrecord-00005-of-01000_266.json -> map_012.bin
324
98
Processing tfrecord-00006-of-00150_134.json -> map_013.bin
16
236
Processing tfrecord-00006-of-01000_162.json -> map_014.bin

### Training

Overfitting one map, you should get completion rate close to 1.

In [12]:
'''
# Run the CLI via the kernel’s interpreter (bypasses ~/.local/bin/puffer)
!{sys.executable} -m pufferlib.pufferl train puffer_drive \
  --env.num-maps 1 \
  --vec.num-envs 4 \
  --vec.num-workers 4 \
  --env.num-agents 64 \
  --train.bptt-horizon 32 \
  --train.batch-size 8192 \
  --train.minibatch-size 1024 \
  --train.max-minibatch-size 1024 \
  --train.update-epochs 2 \
  --train.total-timesteps 1000000 \
  | sed -r 's/\x1B\[[0-9;]*[A-Za-z]//g'
'''

"\n# Run the CLI via the kernel’s interpreter (bypasses ~/.local/bin/puffer)\n!{sys.executable} -m pufferlib.pufferl train puffer_drive   --env.num-maps 1   --vec.num-envs 4   --vec.num-workers 4   --env.num-agents 64   --train.bptt-horizon 32   --train.batch-size 8192   --train.minibatch-size 1024   --train.max-minibatch-size 1024   --train.update-epochs 2   --train.total-timesteps 1000000   | sed -r 's/\x1b\\[[0-9;]*[A-Za-z]//g'\n"

In [13]:
'''
!puffer train puffer_drive \
  --env.num-maps 1 \
  --vec.num-envs 4 \
  --vec.num-workers 4 \
  --env.num-agents 64 \
  --train.bptt-horizon 32 \
  --train.batch-size 8192 \
  --train.minibatch-size 1024 \
  --train.max-minibatch-size 1024 \
  --train.update-epochs 2 \
  --train.total-timesteps 1000000 \
  | sed -r 's/\x1B\[[0-9;]*[A-Za-z]//g'
'''

"\n!puffer train puffer_drive   --env.num-maps 1   --vec.num-envs 4   --vec.num-workers 4   --env.num-agents 64   --train.bptt-horizon 32   --train.batch-size 8192   --train.minibatch-size 1024   --train.max-minibatch-size 1024   --train.update-epochs 2   --train.total-timesteps 1000000   | sed -r 's/\x1b\\[[0-9;]*[A-Za-z]//g'\n"

In [14]:
!{sys.executable} -m pufferlib.pufferl train puffer_drive \
  --env.num-maps 1 \
  --vec.num-envs 4 \
  --vec.num-workers 4 \
  --env.num-agents 64 \
  --train.bptt-horizon 32 \
  --train.batch-size 8192 \
  --train.minibatch-size 1024 \
  --train.max-minibatch-size 1024 \
  --train.update-epochs 2 \
  --train.total-timesteps 1000000 \
  --train.render=0 \
  --train.render-interval=0

/scratch/fz2411/puffer_venv/lib/python3.11/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)
╭──────────────────────────────────────────────────────────────────────────────╮
│  PufferLib 3.0 🐡             CPU: 0.0%  GPU: 0.0%  DRAM: 0.0%   VRAM: 0.0%  │
│                                                                              │
│  Summary          Value    Evaluate      0s   0%    Losses            Value  │
│  Env       p

Train on 64 maps, you should get completion rate close to 0.9.

In [ ]:
!{sys.executable} -m pufferlib.pufferl train puffer_drive \
  --env.num-maps 64 \
  --vec.num-envs 4 \
  --vec.num-workers 4 \
  --env.num-agents 64 \
  --train.bptt-horizon 32 \
  --train.batch-size 8192 \
  --train.minibatch-size 1024 \
  --train.max-minibatch-size 1024 \
  --train.update-epochs 2 \
  --train.total-timesteps 50000000 \
  --train.render=0 \
  --train.render-interval=0

/scratch/fz2411/puffer_venv/lib/python3.11/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)
╭──────────────────────────────────────────────────────────────────────────────╮
│  PufferLib 3.0 🐡             CPU: 0.0%  GPU: 0.0%  DRAM: 0.0%   VRAM: 0.0%  │
│                                                                              │
│  Summary          Value    Evaluate      0s   0%    Losses            Value  │
│  Env       p

In [ ]:
'''
!puffer train puffer_drive \
  --env.num-maps 64 \
  --vec.num-envs 4 \
  --vec.num-workers 4 \
  --env.num-agents 64 \
  --train.bptt-horizon 32 \
  --train.batch-size 8192 \
  --train.minibatch-size 1024 \
  --train.max-minibatch-size 1024 \
  --train.update-epochs 2 \
  --train.total-timesteps 50000000 \
  | sed -r 's/\x1B\[[0-9;]*[A-Za-z]//g'
'''

### Evaluation

- As explained above,
  - Class Drive in **PufferDrive/pufferlib/ocean/drive/drive.py** will only use data from **PufferDrive/pufferlib/resources/drive/binaries**
  - Function process_all_maps() will only process raw json files from **PufferDrive/data/processed/training/** and save them to **PufferDrive/pufferlib/resources/drive/binaries**
  - Therefore, *"puffer eval puffer_drive ..."* will evaluate on the same data
- Below is a temporary workaround that should work, but I have not tested it yet
  - remove everything from **PufferDrive/data/processed/training/** and **PufferDrive/pufferlib/resources/drive/binaries**
  - copy the testing data to **PufferDrive/data/processed/training/**
  - execute **PufferDrive/pufferlib/ocean/drive/drive.py**
  - now **PufferDrive/pufferlib/resources/drive/binaries** should contains the testing data

In [ ]:
!rm data/processed/training/*
!rm resources/drive/binaries/*

In [ ]:
!cp -a GPUDrive_mini/testing/. data/processed/training

In [ ]:
!python pufferlib/ocean/drive/drive.py

In [ ]:
!puffer eval puffer_drive --load-model-path experiments/xxx.pt